# Orchestration par Graphes d'États (Construire son propre LangGraph)

## 1. Introduction

Dans les sessions précédentes, nous avons conçu un micro-framework capable de faire tourner un agent autonome dans une boucle linéaire de type ReAct (Thought $\rightarrow$ Action $\rightarrow$ Observation). Cette architecture est parfaite pour des tâches simples de type "Q&A avec outils".  

Cependant, dès que nous voulons implémenter des processus métier plus complexes, nous faisons face à un mur  :  
- Comment gérer un pipeline où un premier agent rédige un rapport, un second le critique, et le premier le réécrit tant que la critique n'est pas positive?
- Comment modéliser des embranchements déterministes, des exécutions parallèles ou des cycles complexes sans finir avec un code truffé de boucles while imbriquées et de conditions if/else impossibles à maintenir?

Pour résoudre ce problème, l'industrie a adopté le patron de conception du Graphe d'États (State Graph) . Aujourd'hui, nous allons coder notre propre moteur de workflow orienté graphe et l'intégrer à notre dépôt Git ai-agent-lab/.  

## 2. Où en sommes-nous dans le cursus?
- Jour 1 (Hier-soir) : Théorie et analyse comparative des frameworks d'orchestration (*LangChain*, *LangGraph*, *CrewAI*, *ADK*).  

- Jour 2 (Ce matin) : Développement des fondations de notre micro-framework (*Message*, *Conversation*, *ToolRegistry*, *Agent* en boucle linéaire).  

- Jour 3 (Aujourd'hui - Ce module) : Introduction du concept de Graphes d'États. Nous développons notre moteur de workflow graphique (StateGraph) pour créer des agents cycliques et collaboratifs.  

- Jour 4 (Demain) : Migration et réécriture de ce même workflow en utilisant le framework industriel **LangGraph** pour constater la gémellité de nos architectures.

## 3. Objectifs pédagogiques
À la fin de cette journée, vous saurez :
- Modéliser un processus agentique sous la forme d'un graphe dirigé cyclique ou acyclique .
- Concevoir un état partagé centralisé (State) mis à jour de manière prévisible par des fonctions indépendantes .
- Développer un compilateur et un exécuteur de graphe capable de gérer les transitions standards et les routages conditionnels .Implémenter un pattern de correction cyclique (Writer $\rightarrow$ Critic $\rightarrow$ Refiner) .
- Comprendre la mécanique interne de LangGraph et d'ADK 2.0 pour les manipuler sans l'effet "boîte noire" .

## 4. Pourquoi orchestrer par Graphes?
L'orchestration par graphes repose sur une idée simple : **séparer le flux de contrôle de l'implémentation des tâches** .

Dans un script classique, le code décide de la suite :

In [ ]:
doc = rediger_document()
critique = analyser_document(doc)
if critique["valide"]:
    publier(doc)
else:
    corriger(doc) # Où va-t-on ensuite? Comment boucler proprement?

Dans un graphe d'états, nous déclarons une structure de données géométrique:
- Nœuds (Nodes) : Des fonctions Python pures. Chaque nœud reçoit l'état actuel de l'application, effectue un calcul (ou appelle un LLM) et renvoie uniquement les données mises à jour .
- Arêtes (Edges) : Les chemins de transition autorisés. Une arête relie un nœud $A$ à un nœud $B$ .
- Arêtes Conditionnelles (Conditional Edges) : Des aiguillages logiques. Une fonction d'évaluation analyse l'état et retourne le nom du prochain nœud à exécuter .
- L'État (State) : La base de données centrale et partagée par tous les nœuds tout au long du voyage .

## 5. Modélisation mathématique du State Graph

Pour garantir la prévisibilité d'un graphe d'états, nous modélisons son exécution comme un système de transition discret.

Soit le State space $\mathcal{S}$ défini par notre schéma de données. L'état du graphe à l'étape $t$ est un vecteur $S_t \in \mathcal{S}$.

Chaque nœud $N_i \in \mathcal{V}$ (où $\mathcal{V}$ est l'ensemble des sommets du graphe) est une fonction de transition d'état :
$$N_i:\mathcal{S}\rightarrow\mathcal{S}$$

Pour préserver l'immutabilité et la traçabilité, le nœud ne modifie pas l'état en place ; il calcule une mise à jour $\Delta_t$ qui est fusionnée avec l'état précédent via un opérateur de réduction $\oplus$ :
$$S_{t+1}=S_t\oplus\Delta_t$$

Le cheminement entre les nœuds est régi par la fonction d'aiguillage $\delta$ :$$\delta:\mathcal{S}\times\mathcal{V}\rightarrow\mathcal{V}$$

Si le nœud actuel $N_{current}$ est associé à une arête standard, la transition est déterministe. S'il est associé à une arête conditionnelle guidée par une fonction de décision $c:\mathcal{S}\rightarrow\mathcal{K}$ (où $\mathcal{K}$ est un ensemble de clés logiques) et un dictionnaire d'aiguillage $M:\mathcal{K}\rightarrow\mathcal{V}$, la transition vers le nœud suivant $N_{next}$ est calculée ainsi :
$$N_{next}=M(c(S_{t+1}))$$

L'exécution s'arrête lorsque $N_{next}=\emptyset$ ou que l'état de fin formel (END) est atteint.

## 6. 🧠 Notes d'Architecte

**Pourquoi LangGraph s'est-il imposé face au modèle linéaire de LangChain?**
Les premiers frameworks d'agents (notamment LangChain au début de 2023) reposaient sur des structures de "chaînes" rigides. Ces pipelines s'exécutaient de manière strictement linéaire.  

Or, les tâches complexes exigent de la cyclicité (réessayer une action en cas d'échec, corriger un texte, valider une sortie avec un humain). Modéliser un cycle dans un framework linéaire forçait les développeurs à écrire des boucles infinies instables.  

LangGraph a résolu ce problème en remplaçant les chaînes par des graphes d'états explicites . En offrant un contrôle total sur les nœuds et les transitions, il permet de concevoir des systèmes de production déterministes, auditables et hautement résilients .

**Le piège de la sur-ingénierie (The State Machine Sledgehammer)**

Bien que séduisants, les graphes d'états introduisent une surcharge de complexité importante . Définir un schéma d'état, créer des nœuds indépendants, lier les arêtes et compiler le graphe demande trois fois plus de code qu'une simple boucle linéaire .

**La règle d'or d'ingénierie** : Si votre agent n'a pas besoin de cycles complexes, de transitions parallèles ou de validation humaine systématique, n'utilisez pas de graphe . Une simple boucle ReAct ou un script Python de 100 lignes sera beaucoup plus rapide à s'exécuter et plus facile à déboguer .

**Checkpointing et voyage dans le temps (Time Travel)**
L'un des avantages majeurs de l'orchestration par graphe est la capacité d'enregistrer un instantané de l'état $S_t$ avant et après chaque exécution de nœud .

En enregistrant ces snapshots dans une base de données (comme Redis ou PostgreSQL), on obtient deux fonctionnalités critiques pour la production  :

- La durabilité : Si le serveur tombe en panne au milieu d'un processus de 20 étapes, le système peut recharger le dernier état valide et reprendre exactement là où il s'était arrêté .
- Le voyage dans le temps : Lors de la phase de débogage, un développeur peut inspecter l'historique, se positionner à l'étape 5, modifier un paramètre et réexécuter le graphe pour analyser la modification de trajectoire de l'agent .

## 7. 📈 Notes Marché
**La standardisation de l'industrie autour des Graphes d'États**
L'adoption des graphes d'états a atteint un niveau de maturité tel que les géants de la tech calquent désormais leurs architectures sur ce modèle. L'exemple le plus frappant est le lancement récent par Google d'ADK 2.0.  

La version 1.x d'ADK se limitait à de la communication linéaire inter-agents. Avec ADK 2.0, Google a entièrement refondu son framework pour introduire un moteur de workflow par graphes dirigés, validant ainsi la suprématie de ce choix architectural pour les applications d'entreprise d'envergure.  

**Coût vs ROI des architectures de Graphes**
L'implémentation d'un moteur de graphes en production montre les indicateurs économiques suivants au sein des organisations :

|Indicateur technique|Impact financier / ROI|Justification|
| -------- | ------- |------- |
|Temps de développement initial|Hausse de 30 % à 40 %|Nécessite une phase de conception UML et de rigueur d'écriture .|
|Coût de maintenance corrective|Baisse de 70 %|Les pannes sont isolées au sein d'un nœud unitaire facilement testable .|
|Facturation des jetons (Tokens)|Optimisation de 25 %|Les nœuds de décision fins évitent d'envoyer tout l'historique au LLM à chaque étape.|  
|Taux de complétion des tâches complexes|Hausse de 15 % à 50 %|Les boucles de correction automatiques résolvent les erreurs de premier niveau sans intervention humaine .|

## 8. Architecture générale du Workflow
Nous allons créer un workflow de validation rédactionnelle. Un utilisateur demande la rédaction d'un article sur un sujet technique. Notre graphe va coordonner deux nœuds de manière cyclique  :

1. **Nœud "Writer"** : Rédige ou corrige l'article en fonction des retours .

2. **Nœud "Critic"** : Analyse l'article produit et donne une évaluation ("Approuvé" ou "À corriger" avec des retours précis) .

L'arête conditionnelle guidera l'exécution : si le critique valide, on termine (END), sinon on retourne chez le rédacteur .
```
                 +-------------------+
                 |    Entry Point    |
                 +---------+---------+
                           |
                           v
                 +-------------------+
                 |      Writer       |<-------+
                 +---------+---------+        |
                           |                  |
                           v                  | (Si refusé : "corriger")
                 +-------------------+        |
                 |      Critic       |        |
                 +---------+---------+        |
                           |                  |
                           v                  |
                 +---------+---------+        |
                 |  Aiguillage?     |--------+
                 +---------+---------+
                           |
                           | (Si approuvé : "terminer")
                           v
                      +---------+
                      |   END   |
                      +---------+
```

## 2.1. Déclaration de l'État (*mini_framework/state.py*)
Ce fichier définit la structure de l'état partagé par nos nœuds. Nous utilisons une Dataclass pour assurer le typage strict.


## 2.2. Le Moteur de Graphe (*mini_framework/graph.py*)
Ce fichier implémente le compilateur et le runtime d'exécution du graphe d'états. Il gère l'enregistrement des nœuds, le routage et le cycle d'exécution.


## 10. Exemple Pratique d'Exécution : Le Cycle de Rédaction (*examples/writer_critic_graph.py*)
Ce script complet configure et exécute un cycle de rédaction récursif. Pour rester simple et reproductible sans consommer de tokens d'API, nous simulons la logique décisionnelle, ce qui est parfait pour valider l'architecture du graphe .


## 11. Suite de Validation Automatisée (*tests/test_graph.py*)
Pour assurer que notre moteur de graphe ne souffre d'aucune régression logicielle lors des refactorisations futures, voici les tests unitaires à exécuter.  


Pour lancer cette suite de tests :

In [ ]:
pytest tests/test_graph.py

## 12. Exercices & Défis
**Exercice : Ajout de métadonnées de traçabilité**

Objectif : Modifier les nœuds de l'exemple *writer_critic_graph.py* pour enregistrer l'heure système ou l'historique des versions intermédiaires de l'article dans le dictionnaire *metadata* de l'état *AgentState*.  

**Défi : L'Arène de Vote (Consensus Multi-Agents)**
**Objectif** : Concevoir un graphe où deux relecteurs indépendants ("CriticA" et "CriticB") évaluent l'article en parallèle. Un nœud de décision central ("Consensus") reçoit les deux retours d'opinions et applique une règle d'arbitrage majoritaire : l'article n'est validé que si les deux critiques ont retourné le statut d'approbation à *True*.  

## 13. Synthèse
Le Graphe d'États apporte un niveau de contrôle inédit par rapport à une simple boucle ReAct monolithique  :

- Découplage : Chaque nœud est un composant logiciel indépendant, facilitant les tests unitaires et la maintenance corrective .

- Immutabilité : L'état central partagé agit comme l'unique source de vérité logique du système, limitant les dérives comportementales .

- Flexibilité : La structure permet d'implémenter nativement des architectures multi-agents complexes et collaboratives.  

## 14. Bibliographie
- LangGraph Documentation — Conceptual Guides & StateGraph API : (https://langchain-ai.github.io/langgraph/concepts/)   

- Google Cloud Blog — ADK 2.0: Architecting Determinist Workflows : (https://institute.sfeir.com/fr/articles/google-adk-2-agent-development-kit-workflows-multi-agents/)   

- Anthropic Research — Building Reliable Agentic Workflows with Cycles : (https://modelcontextprotocol.io)

## 15. Ce qui sera développé demain
Demain (Semaine 4 — Jour 4), nous allons prendre notre projet rédactionnel Writer/Critic développé aujourd'hui et le migrer intégralement vers LangGraph.  

Vous découvrirez à quel point la structure de l'API de LangGraph (avec leur *StateGraph*, l'enregistrement des nœuds, des transitions directes et des conditional edges) correspond terme pour terme à l'architecture que nous venons de coder aujourd'hui .

Cette transition démontrera l'intérêt profond de notre approche d'ingénierie "From Scratch" : vous ne serez plus jamais de simples utilisateurs d'une librairie tierce complexe, mais des ingénieurs capables de comprendre et de challenger ses choix de conception internes